# Phase 3: The Master ETL Pipeline
**Objective:** Transform massive, imbalanced raw text files into mathematically sound, machine-learning-ready datasets.

**Architecture:**
1. **Extraction (Strict N-1 Logic):** Safely load Fannie Mae text files in chunks. To prevent Look-Ahead Bias, we extract the *final* row ($N$) for Active loans, and the *second-to-last* row ($N-1$) for Prepaid loans. 
2. **Per-Cohort Balancing:** Undersample the majority class to exactly 50/50 *within each specific year* to preserve the economic timeline.
3. **Feature Engineering:** Calculate `REFINANCE_INCENTIVE` and impute missing values with cohort medians.
4. **Encoding & Alignment:** Convert text categories into binary columns (One-Hot Encoding) and perfectly align the Training and Testing datasets so the AI does not crash on unseen variables.

**Temporal Split (Out-of-Time Validation):**
* **Train Set (The Past):** 2000, 2005, 2008, 2013
* **Test Set (The Future):** 2017

In [2]:
import pandas as pd
import numpy as np
import gc
import os

def extract_cohort(file_path, cohort_name, is_test=False):
    """
    Fixed Engine: Applies N-1 Logic, but ONLY balances the Training data.
    Testing data is left in its natural, highly imbalanced state.
    """
    print(f"⏳ Processing {cohort_name}...")
    
    USE_COLS = [1, 2, 7, 9, 12, 15, 22, 23, 25, 26, 27, 28, 29, 30, 43] 
    COL_NAMES = ["LOAN_ID", "ACT_PERIOD", "ORIG_RATE", "UPB", "ORIG_TRM", 
                 "LOAN_AGE", "DTI", "CSCORE_B", "FTHB_FLG", "PURPOSE", 
                 "PROP_TYP", "NUM_UNIT", "OCC_STAT", "STATE", "ZERO_BAL_CODE"]
    
    chunk_iter = pd.read_csv(
        file_path, sep='|', header=None, usecols=USE_COLS, names=COL_NAMES,
        chunksize=150000, low_memory=False, dtype={"LOAN_ID": str, "ZERO_BAL_CODE": str}
    )
    
    processed_chunks = []
    for chunk in chunk_iter:
        chunk_compressed = chunk.groupby("LOAN_ID").tail(2)
        processed_chunks.append(chunk_compressed)
        del chunk
        gc.collect()

    df_concat = pd.concat(processed_chunks)
    df_final_two = df_concat.groupby("LOAN_ID").tail(2).sort_values(by=['LOAN_ID', 'ACT_PERIOD'])
    
    # N-1 Leakage Prevention
    prepaid_codes = ['01', '1', '1.0']
    df_final_two['TARGET_FLAG'] = df_final_two['ZERO_BAL_CODE'].astype(str).str.strip().isin(prepaid_codes).astype(int)
    loan_outcomes = df_final_two.groupby('LOAN_ID')['TARGET_FLAG'].max().reset_index()
    loan_outcomes.rename(columns={'TARGET_FLAG': 'IS_PREPAID'}, inplace=True)
    df_final_two = df_final_two.merge(loan_outcomes, on='LOAN_ID')
    
    df_final_two['ROW_COUNT'] = df_final_two.groupby('LOAN_ID')['LOAN_ID'].transform('count')
    df_final_two['ROW_NUM'] = df_final_two.groupby('LOAN_ID').cumcount() + 1
    
    mask = (df_final_two['ROW_COUNT'] == 1) | \
           ((df_final_two['IS_PREPAID'] == 1) & (df_final_two['ROW_NUM'] == 1)) | \
           ((df_final_two['IS_PREPAID'] == 0) & (df_final_two['ROW_NUM'] == 2))
           
    final_df = df_final_two[mask].drop(columns=['TARGET_FLAG', 'ROW_COUNT', 'ROW_NUM'])
    final_df["COHORT"] = cohort_name
    
    # THE FIX: Only balance if it is Training Data
    if not is_test:
        df_prepaid = final_df[final_df['IS_PREPAID'] == 1]
        df_active = final_df[final_df['IS_PREPAID'] == 0]
        min_class_size = min(len(df_prepaid), len(df_active))
        
        df_balanced = pd.concat([
            df_prepaid.sample(n=min_class_size, random_state=42), 
            df_active.sample(n=min_class_size, random_state=42)
        ]).sample(frac=1, random_state=42)
        
        print(f"✅ {cohort_name} Balanced (Train)! -> Prepaids: {min_class_size:,} | Actives: {min_class_size:,}\n")
        return df_balanced
    else:
        # Leave Testing Data completely alone
        prepaids = final_df['IS_PREPAID'].sum()
        actives = len(final_df) - prepaids
        print(f"✅ {cohort_name} Natural (Test)! -> Prepaids: {prepaids:,} | Actives: {actives:,}\n")
        return final_df

In [3]:
# Define paths to your raw data
RAW_DIR = r"../data/"

train_cohorts = {
    "2000Q1": os.path.join(RAW_DIR, "2000Q1.csv"),
    "2005Q1": os.path.join(RAW_DIR, "2005Q1.csv"),
    "2008Q1": os.path.join(RAW_DIR, "2008Q1.csv"),
    "2013Q1": os.path.join(RAW_DIR, "2013Q1.csv")
}
test_cohort = {"2017Q1": os.path.join(RAW_DIR, "2017Q1.csv")}

print("🚀 ASSEMBLING TRAINING DATA (Balancing applied)...")
train_dfs = []
for name, path in train_cohorts.items():
    if os.path.exists(path):
        train_dfs.append(extract_cohort(path, name, is_test=False))

df_train_raw = pd.concat(train_dfs, ignore_index=True)

print("\n🚀 ASSEMBLING TESTING DATA (Natural Distribution)...")
if os.path.exists(test_cohort["2017Q1"]):
    # THE FIX: We pass is_test=True so it doesn't balance the future
    df_test_raw = extract_cohort(test_cohort["2017Q1"], "2017Q1", is_test=True)

🚀 ASSEMBLING TRAINING DATA (Balancing applied)...
⏳ Processing 2000Q1...
✅ 2000Q1 Balanced (Train)! -> Prepaids: 5,492 | Actives: 5,492

⏳ Processing 2005Q1...
✅ 2005Q1 Balanced (Train)! -> Prepaids: 24,801 | Actives: 24,801

⏳ Processing 2008Q1...
✅ 2008Q1 Balanced (Train)! -> Prepaids: 48,481 | Actives: 48,481

⏳ Processing 2013Q1...
✅ 2013Q1 Balanced (Train)! -> Prepaids: 177,598 | Actives: 177,598


🚀 ASSEMBLING TESTING DATA (Natural Distribution)...
⏳ Processing 2017Q1...
✅ 2017Q1 Natural (Test)! -> Prepaids: 351,247 | Actives: 136,542



In [10]:
import pandas as pd
import numpy as np

def engineer_features(df, train_medians=None):
    df = df.copy()
    
    # 1. Clean Numeric Columns
    numeric_cols = ['ORIG_RATE', 'UPB', 'ORIG_TRM', 'LOAN_AGE', 'DTI', 'CSCORE_B']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    if train_medians is None:
        train_medians = df[numeric_cols].median()
    df[numeric_cols] = df[numeric_cols].fillna(train_medians)
    
    # --- QUICK WIN: Binning Continuous Variables ---
    # We create 5 buckets for Credit Score and 3 for DTI based on density (quantiles)
    # Using duplicates='drop' to prevent errors if too many values are identical
    df['CSCORE_BUCKET'] = pd.qcut(df['CSCORE_B'], q=5, labels=['Very_Poor', 'Poor', 'Fair', 'Good', 'Excellent'], duplicates='drop')
    df['DTI_BUCKET'] = pd.qcut(df['DTI'], q=3, labels=['Low_Debt', 'Mid_Debt', 'High_Debt'], duplicates='drop')
    
    # 2. Dynamic Math & Seasonality
    df['ACT_YEAR'] = df['ACT_PERIOD'].astype(str).str[-4:]
    # Extract month and handle any weird characters
    df['ACT_MONTH'] = pd.to_numeric(df['ACT_PERIOD'].astype(str).str[:-4], errors='coerce').fillna(1).astype(int)
    
    yearly_rates = {'2000': 8.0, '2001': 7.0, '2002': 6.5, '2003': 5.8, '2004': 5.8, '2005': 5.8, 
                    '2006': 6.4, '2007': 6.3, '2008': 6.0, '2009': 5.0, '2010': 4.7, '2011': 4.4, 
                    '2012': 3.6, '2013': 3.9, '2014': 4.1, '2015': 3.8, '2016': 3.6, '2017': 4.0}
    
    df['MARKET_RATE'] = df['ACT_YEAR'].map(yearly_rates).fillna(4.0)
    df['REFINANCE_INCENTIVE'] = df['ORIG_RATE'] - df['MARKET_RATE']
    
    # 3. Robust Boolean Parsing
    df["FTHB_FLG"] = df["FTHB_FLG"].astype(str).str.strip().eq('Y').astype(int)
    
    # 4. Final Cleanup
    cols_to_drop = ["LOAN_ID", "ACT_PERIOD", "ZERO_BAL_CODE", "COHORT", 
                "ACT_YEAR", "MARKET_RATE", "UPB", "ORIG_RATE", "LOAN_AGE", "CSCORE_B", "DTI", "ACT_MONTH"]
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    
    # One-Hot Encoding (Now including our new buckets)
    categorical_cols = ["PURPOSE", "PROP_TYP", "OCC_STAT", "STATE", "CSCORE_BUCKET", "DTI_BUCKET"]
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
    
    return df, train_medians

print("⚙️ Engineering Features and ensuring Data Integrity (with Binning)...")

# Pass 1: Engineer Training Data AND extract its medians
df_train_engineered, training_medians = engineer_features(df_train_raw)

# Pass 2: Engineer Testing Data USING the Training Data's medians
df_test_engineered, _ = engineer_features(df_test_raw, train_medians=training_medians)

# Align columns (Ensures Test set has the exact same columns as Train set)
df_train_final, df_test_final = df_train_engineered.align(df_test_engineered, join='left', axis=1, fill_value=0)

print(f"✅ Engineering Complete! Ready to save.")

⚙️ Engineering Features and ensuring Data Integrity (with Binning)...
✅ Engineering Complete! Ready to save.


In [11]:
# Create the processed directory if it doesn't exist
os.makedirs(r"../data/processed", exist_ok=True)

# Save the final, ML-Ready datasets
train_path = r"../data/processed/train_ready.csv"
test_path = r"../data/processed/test_ready.csv"

df_train_final.to_csv(train_path, index=False)
df_test_final.to_csv(test_path, index=False)

print("\n" + "="*50)
print("🏆 PIPELINE COMPLETE: Data is officially ML-Ready!")
print(f"Train Dataset saved to: {train_path}")
print(f"Test Dataset saved to:  {test_path}")
print("="*50)


🏆 PIPELINE COMPLETE: Data is officially ML-Ready!
Train Dataset saved to: ../data/processed/train_ready.csv
Test Dataset saved to:  ../data/processed/test_ready.csv


In [2]:
import pandas as pd
df69=pd.read_csv(r"C:\Users\shlokui\Desktop\DMproject\data\processed\test_ready.csv")
df69.head(10)

,ORIG_TRM,FTHB_FLG,NUM_UNIT,IS_PREPAID,REFINANCE_INCENTIVE,PURPOSE_P,PURPOSE_R,PURPOSE_U,PROP_TYP_CP,PROP_TYP_MH,...,STATE_WA,STATE_WI,STATE_WV,STATE_WY,CSCORE_BUCKET_Poor,CSCORE_BUCKET_Fair,CSCORE_BUCKET_Good,CSCORE_BUCKET_Excellent,DTI_BUCKET_Mid_Debt,DTI_BUCKET_High_Debt
0,360,0,1,1,0.375,0,1,0,0,0,...,0,0,0,0,1,0,0,0,1,0
1,360,0,2,0,0.875,1,0,0,0,0,...,0,0,0,0,0,0,0,1,1,0
2,240,0,1,1,-0.125,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
3,360,0,1,0,-0.500,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,360,0,1,1,-0.375,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
5,360,0,1,1,0.500,1,0,0,0,0,...,0,0,0,0,0,0,1,0,1,0
6,360,0,1,0,0.750,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,1
7,180,0,1,0,-0.875,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
8,360,1,1,1,0.250,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
9,360,0,1,1,0.125,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1


In [ ]:
df69['IS_PREPAID'].value_counts()

73